## Speed and MSE Benchmarks for FastMatch 

- OLS does regression adjustment.
- Fastmetch with a `scikit` backend does exact nearest neighbor search.
- Fastmatch with a `voyager` backend does approximate nearest neighbor search using the `voyager` library.
- Fastmatch with a `faiss` backend does approximate nearest neighbor search using the `faiss` library.

In [1]:
import time

from scipy.special import expit

import numpy as np
import pandas as pd
from fastmatch import Matching
from sklearn.linear_model import LinearRegression
import warnings

warnings.filterwarnings("ignore")

np.random.seed(42)

In [2]:
def dgp(n=1000, p=5, sigma=1, confounding=True, tau=0):
    X = np.random.normal(0, 2, (n, p))
    # add nonlinear terms
    Xquad = np.hstack((np.log(np.abs(X)), X, X**2, X**3))  # quadratic and cubic terms
    phi = np.random.uniform(-2, 2, p * 4)  # propensity score coefficients
    if confounding:
        propensity = expit(Xquad @ phi)  # logistic function for propensity score
    else:
        propensity = np.full(n, 0.5)
    w = np.random.binomial(1, propensity)
    # Treatment effect
    gamma = np.random.normal(1, 2, p * 4)
    # outcome model is always misspecified w OLS base terms
    y = np.random.rand(1) + Xquad @ gamma + w * tau + np.random.normal(0, sigma, n)
    return y, w, X, Xquad, tau

In [3]:
def onesim(n=10000, p=5, confounding=True):
    y, w, X, Xquad, tau = dgp(n=n, p=p, confounding=confounding)
    # OLS
    start_time = time.time()
    # sklearn has implicit intercept
    ols = LinearRegression()
    ols.fit(np.hstack([X, w.reshape(-1, 1)]), y)
    ols_tau = ols.coef_[-1]
    ols_time = time.time() - start_time

    # Fastmatch (faiss)
    start_time = time.time()
    m_faiss = Matching(estimand="ATT", k=5, bias_corr_mod=None, backend="faiss")
    faiss_tau, _ = m_faiss.fit(y, w, Xquad)
    faiss_time = time.time() - start_time

    # Fastmatch (voyager)
    start_time = time.time()
    m_voyager = Matching(estimand="ATT", k=5, bias_corr_mod=None, backend="voyager")
    voyager_tau, _ = m_voyager.fit(y, w, Xquad)
    voyager_time = time.time() - start_time

    # Fastmatch (scikit)
    start_time = time.time()
    m_scikit = Matching(estimand="ATT", k=5, bias_corr_mod=None, backend="scikit")
    scikit_tau, _ = m_scikit.fit(y, w, Xquad)
    scikit_time = time.time() - start_time
    return {
        "true_tau": tau,
        "ols_tau": ols_tau,
        "faiss_tau": faiss_tau,
        "voyager_tau": voyager_tau,
        "scikit_tau": scikit_tau,
        "ols_time": ols_time,
        "faiss_time": faiss_time,
        "voyager_time": voyager_time,
        "scikit_time": scikit_time,
    }


onesim(1000, 5, True)

{'true_tau': 0,
 'ols_tau': -14.178944583625995,
 'faiss_tau': 20.197968793963927,
 'voyager_tau': 21.516679095473513,
 'scikit_tau': 20.130078468944166,
 'ols_time': 0.0024673938751220703,
 'faiss_time': 0.050343990325927734,
 'voyager_time': 0.03557252883911133,
 'scikit_time': 0.04771828651428223}

In [4]:
def run_simulation(n_sims=100, n=10_000, p=5, confounding=True):
    """Runs a Monte Carlo simulation to compare estimators."""
    results = []
    print(f"Running simulation for {n_sims} iterations...")
    for i in range(n_sims):
        if (i + 1) % 10 == 0:
            print(f"  Iteration {i + 1}/{n_sims}")
        sim_result = onesim(n=n, p=p, confounding=confounding)
        results.append(sim_result)
    return pd.DataFrame(results)

In [5]:
def analyze_results(results_df):
    """Analyzes the results of the Monte Carlo simulation."""
    analysis = {}
    for col in ["ols_tau", "faiss_tau", "voyager_tau", "scikit_tau"]:
        bias = (results_df[col] - results_df["true_tau"]).mean()
        runtime = results_df[col.replace("_tau", "_time")].mean()
        analysis[col.replace("_tau", "")] = {
            "Bias": bias,
            "Runtime (s)": runtime,
        }
    return pd.DataFrame(analysis).T


### low-dim simulation

In [6]:
results = run_simulation(n_sims=100)
analysis = analyze_results(results)
print(analysis)


Running simulation for 100 iterations...
  Iteration 10/100
  Iteration 20/100
  Iteration 30/100
  Iteration 40/100
  Iteration 50/100
  Iteration 60/100
  Iteration 70/100
  Iteration 80/100
  Iteration 90/100
  Iteration 100/100
             Bias  Runtime (s)
ols     -3.046769     0.004759
faiss   -1.343860     0.246936
voyager -2.052221     0.081116
scikit  -2.104894     0.038113


### high-dim simulation

In [7]:
results = run_simulation(n_sims=100, p=20)
analysis = analyze_results(results)
print(analysis)


Running simulation for 100 iterations...
  Iteration 10/100
  Iteration 20/100
  Iteration 30/100
  Iteration 40/100
  Iteration 50/100
  Iteration 60/100
  Iteration 70/100
  Iteration 80/100
  Iteration 90/100
  Iteration 100/100
              Bias  Runtime (s)
ols       2.027995     0.042676
faiss    21.642442     0.568015
voyager  22.107721     0.103634
scikit   21.121810     0.059875


Curse you, curse of dimensionality!

Don't do matching on high-dimensional data. 